# 20 · Deep learning et premiers pas en PyTorch — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch20_deep_learning/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 20.Q1 à 20.Q12, 20.R1 à 20.R3, 20.1 à 20.4, 20.6, 20.7, 20.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 20.13 à 20.17 | tes premiers tenseurs PyTorch, prédire les formes des couches utilitaires, l'autograd sur le réseau minuscule du ch. 18, puis ta librairie regularization.py : dropout, layer norm, pénalité L2 et early stopping (*prochaine session de génération*) | | | |
| B | 20.18 à 20.22 | ta classe nn.Module, Dataset et DataLoader, le zoo des couches, le plan de VGG16, la batchnorm de ta librairie (*prochaine session de génération*) | | | |
| C | 20.23 à 20.27 | la boucle d'entraînement complète sur MNIST, une chasse aux bugs, des courbes d'entraînement et de validation, un modèle ImageNet face à l'inconnu, la sauvegarde (*prochaine session de génération*) | | | |
| D | 20.28 à 20.31 | dropout, batchnorm, AdamW et early stopping sur Fashion-MNIST, une ablation, un réseau de 30 couches, le défi des 90 % (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Prévoir la forme d'un tenseur couche après couche, et dimensionner la sortie d'un réseau.
- Calculer à la main un dropout inversé, une batchnorm (entraînement et évaluation) et une layer norm, puis les programmer dans `mylearn.nn.regularization` et les retrouver dans PyTorch.
- Écrire un `nn.Module`, un `DataLoader` et une boucle d'entraînement PyTorch, et mener une ablation de régularisation.

**Rappel express.** Dropout inversé : $\tilde a = a\,m/(1 - p)$, $m \sim \mathrm{Bernoulli}(1 - p)$, un nouveau masque à chaque passe avant, l'identité en évaluation. Batchnorm : $\hat x = (x - \mu_B)/\sqrt{\sigma_B^2 + \epsilon}$ (variance biaisée), $y = \gamma\hat x + \beta$ ; statistiques glissantes $\mu^{\text{run}} \leftarrow (1 - \rho)\,\mu^{\text{run}} + \rho\,\mu_B$ avec $\rho = 0{,}1$ et la variance non biaisée. Layer norm : la même formule, sur les features de chaque exemple. Pénalité L2 : $\frac{\lambda}{2}\sum\lVert\mathbf{W}\rVert^2$, de gradient $\lambda\mathbf{W}$. Boucle PyTorch : `model.train()`, puis pour chaque mini-batch `logits = model(X)`, `loss = loss_fn(logits, y)`, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` ; `model.eval()` et `torch.no_grad()` pour mesurer.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 20.1 à 20.4, 20.6, 20.7 et 🧮 20.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une forme : entre guillemets (`"(3, 5)"`, ou `"(7,)"` pour une seule dimension). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les preuves ∂ 20.5 et 20.8, la réflexion (🗣️ 20.10, ⚖️ 20.11, 📄 20.12) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np

# Ex 20.3: inverted dropout on five activations, keep a neuron when u >= p
A_203 = np.array([2.0, -1.0, 0.5, 3.0, 4.0])
U_203 = np.array([0.1, 0.7, 0.35, 0.9, 0.5])
P_203 = 0.4
MASK_203 = (U_203 >= P_203) / (1 - P_203)
OUT_203 = A_203 * MASK_203

# Ex 20.4: batch normalization of a mini-batch of four examples, eps = 0
X_204 = np.array([[1.0, 10.0], [3.0, 14.0], [5.0, 6.0], [7.0, 10.0]])
GAMMA_204, BETA_204, MOMENTUM_204 = np.array([2.0, 0.5]), np.array([0.0, 1.0]), 0.1
MU_204, VAR_204 = X_204.mean(axis=0), X_204.var(axis=0)                     # biased variance
XHAT_204 = (X_204 - MU_204) / np.sqrt(VAR_204)
Y_204 = GAMMA_204 * XHAT_204 + BETA_204
RUN_MEAN_204 = (1 - MOMENTUM_204) * np.zeros(2) + MOMENTUM_204 * MU_204
RUN_VAR_204 = (1 - MOMENTUM_204) * np.ones(2) + MOMENTUM_204 * X_204.var(axis=0, ddof=1)   # unbiased
EVAL_204 = GAMMA_204 * (np.array([4.0, 12.0]) - RUN_MEAN_204) / np.sqrt(RUN_VAR_204) + BETA_204

# Ex 20.6: the parameters of VGG16
VGG_BLOCKS = [[64, 64], [128, 128], [256, 256, 256], [512, 512, 512], [512, 512, 512]]
VGG_CONV = []
_c_in = 3
for _block in VGG_BLOCKS:
    for _c_out in _block:
        VGG_CONV.append((9 * _c_in + 1) * _c_out)
        _c_in = _c_out
VGG_FC = [25088 * 4096 + 4096, 4096 * 4096 + 4096, 4096 * 1000 + 1000]
VGG_TOTAL = sum(VGG_CONV) + sum(VGG_FC)

# Ex 20.7: layer norm of one example, eps = 0
X_207 = np.array([1.0, 2.0, 3.0, 6.0])
LN_207 = (X_207 - X_207.mean()) / np.sqrt(X_207.var())

# Ex 20.9: training VGG16 on ImageNet
TRAIN_OPS_209 = 3 * 15.5e9                       # forward + backward, per image
TOTAL_OPS_209 = 1.28e6 * 74 * TRAIN_OPS_209

**20.Q1 — Deep learning et machine learning : qui contient qui ?**

In [3]:
wb.record("20.Q1a", True, mistakes={"relis le §20.2 de la fiche : quel est l'usage le plus courant ?": False})
wb.record("20.Q1b", False, mistakes={"le livre signale que certains auteurs réservent « machine learning » aux méthodes sans réseaux": True})
wb.record("20.Q1c", "C", mistakes={"un MLP à deux couches cachées est une pile de couches": "A",
          "un réseau de convolution est un réseau de neurones à plusieurs couches": "B",
          "un réseau récurrent est un réseau de neurones": "D"})
wb.record("20.Q1d", "B", mistakes={"la hiérarchie ne porte pas sur le nombre de neurones, mais sur ce que chaque couche reçoit": "A",
          "rien dans le réseau ne trie les classes par fréquence": "C",
          "les poids ne sont pas triés : qu'est-ce que chaque couche reçoit en entrée ?": "D"})

WB_ANSWER {"hash": "fd4a6e0b0c328fa6a398256f41624e453aa2b859c2487fc3dbdb084d30ac05d9", "id": "20.Q1a", "kind": "bool", "mistakes": {"8cc886e99ec1013d3c0c53d0a96733486019408c5d1f3c1a3eb57e95449af2cb": "relis le §20.2 de la fiche : quel est l'usage le plus courant ?"}}
📝 Ex 20.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "33609730f43e8017ccec22651a0f9dbf43442bf84fd30c59e780371674f70ee1", "id": "20.Q1b", "kind": "bool", "mistakes": {"3bd1caad9ea604b0d4886659d90f9718c7e139084554e222b49bd38338bd91a4": "le livre signale que certains auteurs réservent « machine learning » aux méthodes sans réseaux"}}
📝 Ex 20.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "9f818f04b05ea427e2656521d44b30e9d8d7f12dcd9cc177f87c7324b7a2aa7d", "id": "20.Q1c", "kind": "str", "mistakes": {"0bacb4dcaecda0fd1fa7429b5a695a6de47d5fa71a2af8816d8b1a6bc583f755": "un réseau récurrent est un réseau de neurones", "0cf642830262ce6e793a2237d619a07e49f79fb4f58e995a74fbbc19c0a05ade": "un réseau de convolution est un 

**20.Q2 — Tenseurs : nombre de dimensions, taille, forme**

In [4]:
wb.record("20.Q2a", 4, mistakes={"compte les nombres de la forme, pas les dimensions d'une image": 3,
          "le nombre de dimensions est la longueur de la forme": 32})
wb.record("20.Q2b", 32 * 3 * 28 * 28, mistakes={"le nombre d'éléments est le PRODUIT des tailles": 32 + 3 + 28 + 28,
          "multiplie aussi par le nombre de canaux, 3": 32 * 28 * 28,
          "multiplie aussi par la taille du mini-batch, 32": 3 * 28 * 28})
wb.record("20.Q2c", "(3, 28, 28)", mistakes={"x[0] retire seulement la première dimension, celle des exemples": "(28, 28)",
          "x[0] retire la première dimension : il reste trois nombres": "(32, 3, 28)",
          "x[0] garde les canaux, puis la hauteur et la largeur": "(28, 28, 3)",
          "la bonne forme, mais écris-la comme un tuple, sans torch.Size : (3, 28, 28)": "torch.Size([3, 28, 28])",
          "la bonne forme, mais écris-la entre parenthèses, comme un tuple : (3, 28, 28)": "[3, 28, 28]",
          "c'est le même tuple, mais écris-le sans la virgule finale, comme PyTorch l'affiche : (3, 28, 28)": "(3, 28, 28,)"})
wb.record("20.Q2d", True, mistakes={"relis la définition du livre (§20.2.1) : un tenseur est-il « troué » ?": False})
wb.record("20.Q2e", "B", mistakes={"float64 est le type par défaut de NumPy, pas de PyTorch": "A",
          "1.0 est un réel : un tenseur d'entiers viendrait de torch.tensor([1, 2])": "C",
          "float16 n'est jamais un type par défaut": "D"})

WB_ANSWER {"hash": "e56b7e79e10303d28cb971e3bffc23c0a55de784fc3a2f3eb9ae88e4a8a60bbd", "id": "20.Q2a", "kind": "int", "magnitude": 0, "mistakes": {"01c8adbe4f6faed50ee44fff19c1ac7dc65ae5da47bdfed2d428d61b69ce36c4": "compte les nombres de la forme, pas les dimensions d'une image", "a588e96e5f3e701ae299551f3e17ad3c094fd056de209b56ef1eb43f4660b4b2": "le nombre de dimensions est la longueur de la forme"}, "sign": 1}
📝 Ex 20.Q2a : réponse enregistrée (int).
WB_ANSWER {"hash": "175cad84f02bf746daa4064c4b4e76ee73a9fa560a40ecd8b095f8753d931ed4", "id": "20.Q2b", "kind": "int", "magnitude": 4, "mistakes": {"56b92f5082e4a34bf1156b8bbfb1afbc7790da2561736291a4cefaae47c6ee86": "multiplie aussi par le nombre de canaux, 3", "83fe666e532a82867bd37555313573503df95acf6f1545bc3759edefa5c1af7b": "multiplie aussi par la taille du mini-batch, 32", "fcea31ba8ccb6350fd14b26fc22559519e572a04e65198e1f18b1433e1543002": "le nombre d'éléments est le PRODUIT des tailles"}, "sign": 1}
📝 Ex 20.Q2b : réponse enregistré

**20.Q3 — Couche d'entrée, couche de sortie : laquelle contient des neurones ?**

In [5]:
wb.record("20.Q3a", False, mistakes={"relis le §20.3.1 : la couche d'entrée calcule-t-elle quelque chose ?": True})
wb.record("20.Q3b", True, mistakes={"qui produit les scores de sortie, sinon des neurones ?": False})
wb.record("20.Q3c", 2, mistakes={"la couche d'entrée et la couche de sortie ne sont pas cachées": 4,
          "seules les couches entre l'entrée et la sortie sont cachées : la sortie n'en est pas une": 3,
          "784 → 256 → 128 → 10 : combien de couches entre l'entrée et la sortie ?": 1})
wb.record("20.Q3d", "B", mistakes={"le livre dessine les entrées en bas : où est le haut ?": "A",
          "« plus haute » parle de position dans la pile, pas de taille": "C",
          "toutes les couches sont entraînées en même temps": "D"})

WB_ANSWER {"hash": "f4c3ce4370ea5afec8db2455d4fff434c961141a39d413068cddba68bae0e011", "id": "20.Q3a", "kind": "bool", "mistakes": {"05fe71d7a98f2276d5b34bcc8cc609f52294974bf72b35e2467ded978ad61c67": "relis le §20.3.1 : la couche d'entrée calcule-t-elle quelque chose ?"}}
📝 Ex 20.Q3a : réponse enregistrée (bool).
WB_ANSWER {"hash": "1147381c2738dab9beebad7dabf2512301e1d2d8cb6f8a68173b2bf6045a40d7", "id": "20.Q3b", "kind": "bool", "mistakes": {"235cc11de49d52a1801531b09e33b912d037ac5e61121567c088e2f84534f224": "qui produit les scores de sortie, sinon des neurones ?"}}
📝 Ex 20.Q3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "4571d74d23b04ba8a4bf0b970fe82cd58d9a3f21a62230919bc4797a4b91225d", "id": "20.Q3c", "kind": "int", "magnitude": 0, "mistakes": {"a0d054d96f5142fa0481a0e1a83daf4177731c8cd89d7c5a551cda69ef7c66f4": "la couche d'entrée et la couche de sortie ne sont pas cachées", "a1e3965d4c48802e4c8fdd317c75c4dbcfc108d57aa17016e139a46a1e4e1d10": "784 → 256 → 128 → 10 : combien de c

**20.Q4 — Couche dense et couche d'activation séparée**

In [6]:
wb.record("20.Q4a", 4 * 3, mistakes={"sans les biais : un poids par connexion": 15, "chaque neurone reçoit les 4 sorties": 7})
wb.record("20.Q4b", 4 * 3 + 3, mistakes={"un biais par neurone, pas par entrée": 16, "ajoute les 3 biais": 12})
wb.record("20.Q4c", True, mistakes={"relis le §20.4.2 : une activation dans la couche ou juste après, est-ce différent ?": False})
wb.record("20.Q4d", "B", mistakes={"CrossEntropyLoss applique elle-même log_softmax : une softmax avant serait de trop": "A",
          "la loss a besoin de tous les scores, pas seulement de la classe gagnante": "C",
          "la sigmoïde va avec BCEWithLogitsLoss, et encore : celle-ci la fait elle-même": "D"})

WB_ANSWER {"hash": "1338773e6fe4cb1b8d100d3bfe4758cd4d24b065e3ffbc0fbd07912df2261f5e", "id": "20.Q4a", "kind": "int", "magnitude": 1, "mistakes": {"4992676afe3fa2d8f3dc5bf4f3a24b4cc55e663406733fcbf3ae135acc07c52b": "sans les biais : un poids par connexion", "bbeb98d20a784e9294296bd57cd40b073541245e276cea9b3d739da45a472248": "chaque neurone reçoit les 4 sorties"}, "sign": 1}
📝 Ex 20.Q4a : réponse enregistrée (int).
WB_ANSWER {"hash": "8365df667364ab3424b08895850a347a1c613a8c33dc33f38c6896fc7dfff9d8", "id": "20.Q4b", "kind": "int", "magnitude": 1, "mistakes": {"73e6852df2f4a07136799ad838ca9499b719dea4bdb7ab4bbc1a7e60b8881781": "ajoute les 3 biais", "f23724b85652d6a231f49b017dfcdf836c257bff6b4ca31b41aab42cc97d5bca": "un biais par neurone, pas par entrée"}, "sign": 1}
📝 Ex 20.Q4b : réponse enregistrée (int).
WB_ANSWER {"hash": "925538fc586143f30799a3dc8e5c7c1e840e6648e1c9e0877408b02342bed99a", "id": "20.Q4c", "kind": "bool", "mistakes": {"ae669a2458d531e79f2a41630c2df18cefe96845084d52f4ddb

**20.Q5 — Vrai ou faux : le dropout tire ses neurones une fois par epoch**

In [7]:
wb.record("20.Q5a", False, mistakes={"c'est ce que dit le livre ; relis l'encadré « Le livre, corrigé » du §20.4.3": True})
wb.record("20.Q5b", 500, mistakes={"un nouveau masque à chaque passe avant, donc à chaque mini-batch": 1})
wb.record("20.Q5c", 1 / (1 - 0.2), decimals=2, mistakes={"on divise par 1 − p, on ne multiplie pas": 0.8, "on divise par 1 − p, pas par p": 5.0})
wb.record("20.Q5d", "C", mistakes={"en évaluation, le dropout ne coupe plus rien": "A",
          "c'est la règle de l'article d'origine, appliquée aux poids ; le dropout inversé n'a rien à corriger au test": "B",
          "en évaluation, aucun neurone n'est coupé": "D"})

WB_ANSWER {"hash": "96f31816619629aa1dea1a4a63b451408c447ce32cec961abfdcbc757f5e043e", "id": "20.Q5a", "kind": "bool", "mistakes": {"4dc371ce3aa9318485e3c9a02fae9050663ba5c128af47e4b6c5536a45390ef0": "c'est ce que dit le livre ; relis l'encadré « Le livre, corrigé » du §20.4.3"}}
📝 Ex 20.Q5a : réponse enregistrée (bool).
WB_ANSWER {"hash": "0d291898acfb5ef86e5f3c9fefd6c980532e5b64780b70b950a1dec749cef36c", "id": "20.Q5b", "kind": "int", "magnitude": 2, "mistakes": {"017a86a6264ef2062cc951ea4c577b91d75582d5d2f2cc6ee9d677d2213d092c": "un nouveau masque à chaque passe avant, donc à chaque mini-batch"}, "sign": 1}
📝 Ex 20.Q5b : réponse enregistrée (int).
WB_ANSWER {"alt_coarse": ["5bc9bedc7ee095d5cdc5a26d4b4286465cbf2cfc343e0a49a24939a9b9a50dbd"], "decimals": 2, "hash": "3b1f303a4d4029e1303f7f0e8cc7bdba21c0c7e8dbcae424ae850ff15cb0d991", "hash_coarse": "c5b74d3c2a915ad28579c67c3141e44267bd78ca9f11481e4a8e9807b5581ac6", "id": "20.Q5c", "kind": "float", "magnitude": 0, "mistakes": {"3f3f4e5d9

**20.Q6 — Vrai ou faux : la batchnorm n'a aucun paramètre**

In [8]:
wb.record("20.Q6a", False, mistakes={"c'est ce que dit le livre ; relis l'encadré « Le livre, corrigé » du §20.4.4": True})
wb.record("20.Q6b", 2 * 64, mistakes={"γ ET β : deux paramètres par feature": 64,
          "les statistiques glissantes ne sont pas apprises par la rétropropagation": 4 * 64})
wb.record("20.Q6c", 2 * 64, mistakes={"une moyenne ET une variance par feature": 64,
          "on compte les valeurs stockées : une moyenne et une variance pour chacune des 64 features": 2})
wb.record("20.Q6d", "B", mistakes={"en évaluation, une prédiction ne doit pas dépendre des autres exemples du mini-batch": "A",
          "la batchnorm normalise aussi en évaluation, avec d'autres statistiques": "C",
          "les statistiques glissantes sont mises à jour à chaque mini-batch d'entraînement": "D"})

WB_ANSWER {"hash": "952d4009581785c535312c91fb794a7d517933554090c26adb6719b62c5904f4", "id": "20.Q6a", "kind": "bool", "mistakes": {"cb3e918620d0e8e6267b1f99033b13b045fc7aa3105c93cc579d91950d35f128": "c'est ce que dit le livre ; relis l'encadré « Le livre, corrigé » du §20.4.4"}}
📝 Ex 20.Q6a : réponse enregistrée (bool).
WB_ANSWER {"hash": "5c4fdbd2d08b85673d2eaa83eef989eb6e0053f1bc87af8a80d6aa74241912f0", "id": "20.Q6b", "kind": "int", "magnitude": 2, "mistakes": {"46724577bd437947ecc8e7557595d457b5f3b17adb8fa60c7171ab19c019dd3d": "les statistiques glissantes ne sont pas apprises par la rétropropagation", "5ecdee684a56386100f3fad1b6da40308e9ba1effa545674fc8dfe8993e7903d": "γ ET β : deux paramètres par feature"}, "sign": 1}
📝 Ex 20.Q6b : réponse enregistrée (int).
WB_ANSWER {"hash": "f2237890b62b64f5b0a5651932a07df37c3905667db5f3083aab9f1baceac04f", "id": "20.Q6c", "kind": "int", "magnitude": 2, "mistakes": {"75528633476ec7ff98f06982efc85504c6eca89c038d98c7ae16210d575b9f55": "une moyen

**20.Q7 — Convolution et pooling en un coup d'œil**

In [9]:
wb.record("20.Q7a", 2 * (3 * 3 * 1 + 1), mistakes={"ajoute un biais par filtre": 18, "un biais par filtre, pas un par poids": 36,
          "deux filtres : multiplie par 2": 10})
wb.record("20.Q7b", 512 // 2, mistakes={"des blocs de 2 × 2 : on divise la largeur par 2": 128})
wb.record("20.Q7c", 30 // 3, mistakes={"des blocs de 3 × 3 disjoints : on divise par 3": 28})
wb.record("20.Q7d", True, mistakes={"un filtre 3 × 3 déborde d'un pixel de chaque côté : que comble un padding de 1 ?": False})

WB_ANSWER {"hash": "3dedb30f7ea18de290ff463a45ba8235a9c38a95273b11a2b9a6da0dc769d8bc", "id": "20.Q7a", "kind": "int", "magnitude": 1, "mistakes": {"155e55ecca4729157e4f6650dab0240cf972467589e9be51e6ebb798ed3321d9": "ajoute un biais par filtre", "1e19ccccff6fa01a14d532d8d7d2c7458bfda187298ea516ab419d2623a5a2bc": "deux filtres : multiplie par 2", "5c6a104a1eb8c9ce3294f905f43f678dc0726a24c1763e23ec1c64ff47fb6d11": "un biais par filtre, pas un par poids"}, "sign": 1}
📝 Ex 20.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "88ebd327d9c7d08111f97c5cc6edaf428b894f2db573fb9bff06ea1a3d5d79a0", "id": "20.Q7b", "kind": "int", "magnitude": 2, "mistakes": {"411ec10312854f7ed492d9dcbac48f814cc3de4eefc8195220c29a7852450c70": "des blocs de 2 × 2 : on divise la largeur par 2"}, "sign": 1}
📝 Ex 20.Q7b : réponse enregistrée (int).
WB_ANSWER {"hash": "9c7a2e283767d3bfe871d2a81018fc8390320786b42962bbeea14fb125c29e2e", "id": "20.Q7c", "kind": "int", "magnitude": 1, "mistakes": {"5b7d4fe42a0d9bbae4ba5aa1

**20.Q8 — Récurrence n'est pas récursion**

In [10]:
wb.record("20.Q8a", False, mistakes={"relis le §20.4.7 : le livre insiste sur la différence": True})
wb.record("20.Q8b", "B", mistakes={"une photo se classe d'un coup, sans ce qui précède": "A",
          "les features d'une maison arrivent toutes ensemble": "C",
          "une image se traite d'un coup": "D"})
wb.record("20.Q8c", True, mistakes={"une couche dense garde-t-elle un état d'une entrée à l'autre ?": False})

WB_ANSWER {"hash": "b4350e80de52fcc0a7b1aa611aee223d01b127669ab8195ef4fce898d19dfa14", "id": "20.Q8a", "kind": "bool", "mistakes": {"d0ff48adc6674446dbd1edcddfad33b079d6c06d2988f34173167c21602e7972": "relis le §20.4.7 : le livre insiste sur la différence"}}
📝 Ex 20.Q8a : réponse enregistrée (bool).
WB_ANSWER {"hash": "6b4aa42234d3db5981108653dc90d0d61d3f177d9c78d4a68405f5e17e7de632", "id": "20.Q8b", "kind": "str", "mistakes": {"31166e66f3e3b91e84a76cefcd6264a0c828b96c27e3765bc6cb387557862ac8": "une photo se classe d'un coup, sans ce qui précède", "bb7679b16b4a9db584e2e1921649b83adc980da3f283ab1e7548fd43995a2810": "une image se traite d'un coup", "d128839a0c9356574f24254ba80462707a596054a64fefcc063bc1acd8bf289e": "les features d'une maison arrivent toutes ensemble"}}
📝 Ex 20.Q8b : réponse enregistrée (str).
WB_ANSWER {"hash": "deb759e6e479b27e2465fa5ee8425d4d28d8af5b4b8c6e6b247228f6beaedeef", "id": "20.Q8c", "kind": "bool", "mistakes": {"ded37f3dc04cfc30dc6791d635dfb1edd13dba33cbbea5559

**20.Q9 — Aplatir, remodeler, rogner, rembourrer, suréchantillonner, bruiter**

In [11]:
wb.record("20.Q9a", 8 + 2 * 1, mistakes={"une bordure de 1 de CHAQUE côté": 9})
wb.record("20.Q9b", 7 * 2, mistakes={"le suréchantillonnage agrandit": 3})
wb.record("20.Q9c", (28 - 24) // 2, mistakes={"28 − 24 = 4 pixels en tout, répartis sur deux côtés": 4})
wb.record("20.Q9d", "D", mistakes={"aplatir change la forme, sans rien apprendre": "A",
          "le suréchantillonnage par défaut répète les valeurs, sans rien apprendre": "B",
          "le zero-padding ajoute des 0 fixes": "C"})
wb.record("20.Q9e", False, mistakes={"comme le dropout, une couche de bruit est une régularisation : quand doit-elle agir ?": True})

WB_ANSWER {"hash": "5426b5a18f9609b8bb03443cbd1bca06b35d8d1536f83f34aab08701df242e12", "id": "20.Q9a", "kind": "int", "magnitude": 1, "mistakes": {"090d69084d61ac04ba18c0826f278f1e95cb182afb44be2d34688ba9c992ab1b": "une bordure de 1 de CHAQUE côté"}, "sign": 1}
📝 Ex 20.Q9a : réponse enregistrée (int).
WB_ANSWER {"hash": "156fe7e807e7f38dbbf865bf9b742ebae17844fa0fd621a2dae49ebbd980a26f", "id": "20.Q9b", "kind": "int", "magnitude": 1, "mistakes": {"2da6b4a882b5ce8fabe7ec72acdc41fe8d012da3d0d581886ade654b48a70e13": "le suréchantillonnage agrandit"}, "sign": 1}
📝 Ex 20.Q9b : réponse enregistrée (int).
WB_ANSWER {"hash": "1117942d9a0050b65d07f1a3a205ba23709b009453506df1eb98adaf3d0c1488", "id": "20.Q9c", "kind": "int", "magnitude": 0, "mistakes": {"472f07a4078368a1a8d7e2afe0e9408636affb0be9c9bdf1a679de6289aa09c5": "28 − 24 = 4 pixels en tout, répartis sur deux côtés"}, "sign": 1}
📝 Ex 20.Q9c : réponse enregistrée (int).
WB_ANSWER {"hash": "f6cc6623a7bccef56a604adedc40f555b9e1f38879f4c241ab30

**20.Q10 — VGG16 : ce qu'il reconnaît et ce qu'il ignore**

In [12]:
wb.record("20.Q10a", 13, mistakes={"16 couches de calcul en tout, dont 3 denses": 16})
wb.record("20.Q10b", 3, mistakes={"le pooling et l'aplatissement ne sont pas des couches denses": 5})
wb.record("20.Q10c", 1000, mistakes={"relis le §20.6 : le nombre de classes d'ImageNet utilisées au concours": 100})
wb.record("20.Q10d", "B", mistakes={"relis l'encadré « Le livre, corrigé » du §20.6 : quelle erreur le concours mesure-t-il ?": "A",
          "le concours mesure l'erreur sur des images qui ne servent pas à l'entraînement": "C",
          "c'est une accuracy, pas une précision": "D"})
wb.record("20.Q10e", False, mistakes={"sa couche de sortie a 1 000 neurones : combien de noms peut-il donner ?": True})

WB_ANSWER {"hash": "ebc854d74a7e7f6cf8933c838f269413257bb94a822a1108d8721de27b6c5bbb", "id": "20.Q10a", "kind": "int", "magnitude": 1, "mistakes": {"e104048575d5d59af6b861a9f50f04c97c089edd3ca80d9aefa55ccb4156ed5d": "16 couches de calcul en tout, dont 3 denses"}, "sign": 1}
📝 Ex 20.Q10a : réponse enregistrée (int).
WB_ANSWER {"hash": "8d3be3ef177677229ed62c0feb7dc12c5722645694ef1330a605f27b71e11401", "id": "20.Q10b", "kind": "int", "magnitude": 0, "mistakes": {"25bab0430c6499e4c5d860011ea0ae9c4f9d1c8bdc72485337c3accd8e643a5d": "le pooling et l'aplatissement ne sont pas des couches denses"}, "sign": 1}
📝 Ex 20.Q10b : réponse enregistrée (int).
WB_ANSWER {"hash": "94110dba10dc934d8ceab4418fb7f2df23f7a669872f3ea58a709f0b7851b0e6", "id": "20.Q10c", "kind": "int", "magnitude": 3, "mistakes": {"5c4cf88c5c97bd2c85f68c067c6f1a56af86fe2997c1f0f4cbb2bfcf6a0bc674": "relis le §20.6 : le nombre de classes d'ImageNet utilisées au concours"}, "sign": 1}
📝 Ex 20.Q10c : réponse enregistrée (int).
WB_AN

**20.Q11 — Construire un réseau profond : par où commencer ?**

In [13]:
wb.record("20.Q11a", "B", mistakes={"avant l'architecture, il faut connaître les données": "A",
          "on règle le learning rate une fois le réseau choisi": "C",
          "un grand réseau sur des données qu'on n'a pas regardées fait perdre du temps": "D"})
wb.record("20.Q11b", "C", mistakes={"relis le §20.7 : le livre nomme un hyperparamètre de l'optimiseur": "A",
          "la taille des mini-batches compte, mais le livre en nomme un autre": "B",
          "une graine ne s'optimise pas": "D"})
wb.record("20.Q11c", False, mistakes={"relis le §20.7 : le livre parle d'expérience et d'essais": True})
wb.record("20.Q11d", "B", mistakes={"un MLP est fait de couches denses": "A", "un RNN est fait de couches récurrentes": "C",
          "un GAN est une association de deux réseaux (ch. 27)": "D"})

WB_ANSWER {"hash": "e308641b308a16a58be35c4b88964215f5614f079945d3d4341b7d60a4be861c", "id": "20.Q11a", "kind": "str", "mistakes": {"766713b6d946c79e40712715b729b90887fc578b366d4c87abb95d0ac0148f91": "un grand réseau sur des données qu'on n'a pas regardées fait perdre du temps", "ad905fe7f1a36b3f218e6afd4929b6d1860d8e393411c9c69157e6f1bb92d02c": "on règle le learning rate une fois le réseau choisi", "be0d77e1e0d605e2877a9a90b9e7e07b72df49b1ef873b7fc4150ab9f9a6b800": "avant l'architecture, il faut connaître les données"}}
📝 Ex 20.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "cffac2fc30d00b95abb29461667468c30e55869d6e84dae0afb98ccf2dd01b55", "id": "20.Q11b", "kind": "str", "mistakes": {"7b359bc388d4def04fd3226334d3a1f5ed9b6322a0c5b47ab1701c2ddfb9272d": "une graine ne s'optimise pas", "9f32e6e529178a964f17cb0883ea319c16e4754bd853cfb4b71125f82895e78c": "relis le §20.7 : le livre nomme un hyperparamètre de l'optimiseur", "ad1a1cd19a3aab5444d560d99974f51acebd74188e849c42b1a07d56141b1

**20.Q12 — Explication complète ou explication satisfaisante ?**

In [14]:
wb.record("20.Q12a", False, mistakes={"relis le §20.8.1 : l'exemple des chaussettes bleues est complet, est-il satisfaisant ?": True})
wb.record("20.Q12b", "B", mistakes={"le volume des données n'empêche pas de vérifier": "A",
          "le nombre de couches gêne l'explication, pas la vérification": "C",
          "rien n'oblige la banque à jeter les dossiers : que lui manque-t-il pour les refusés ?": "D"})
wb.record("20.Q12c", "A", mistakes={"une boîte blanche est au contraire un modèle qu'on peut lire": "B",
          "une régression linéaire s'explique très bien par ses coefficients": "C",
          "la dérive est un changement du monde après l'entraînement": "D"})
wb.record("20.Q12d", False, mistakes={"relis le §20.8.1 : le monde a-t-il pu changer en trois ans ?": True})

WB_ANSWER {"hash": "61f7110437148bc9a4c582d54b025141b011d52eb1c11b7e4e93a329168628c9", "id": "20.Q12a", "kind": "bool", "mistakes": {"5d23faa79238f03678ce54ee63164ddb656081e00b9f08a6753d30ae158416b8": "relis le §20.8.1 : l'exemple des chaussettes bleues est complet, est-il satisfaisant ?"}}
📝 Ex 20.Q12a : réponse enregistrée (bool).
WB_ANSWER {"hash": "878a0d939773c287295c1939fe958f279a7f518056a2ad9ef41ce2e9a5dafd49", "id": "20.Q12b", "kind": "str", "mistakes": {"4f8df6be6cd97d628d10cfb1f380ea456b93654053f8276ffc092924396c83c0": "rien n'oblige la banque à jeter les dossiers : que lui manque-t-il pour les refusés ?", "552521e30129b98b07944e2620bab4661a59fb7883a487609fd11cc8e6c4af28": "le nombre de couches gêne l'explication, pas la vérification", "f7fd9024e805c22cd24bc198333aff33c890914bd488efbd56231bd87ca612bf": "le volume des données n'empêche pas de vérifier"}}
📝 Ex 20.Q12b : réponse enregistrée (str).
WB_ANSWER {"hash": "db9848466873904a926fdb78dbe6cb572dbad3fe9614e68b6e743a8d9926d3

**20.R1 — Que fait weight_decay dans AdamW ?**

In [15]:
wb.record("20.R1a", 2 * (1 - 0.01 * 0.1), decimals=3, mistakes={"le facteur est 1 − η λ, avec η = 0,01 : multiplie λ par η": 1.8,
          "le gradient est nul, mais le weight decay agit quand même": 2.0})
wb.record("20.R1b", False, mistakes={"c'est la différence entre Adam(weight_decay=…) et AdamW : relis la fiche du ch. 19": True})
wb.record("20.R1c", 0.01, decimals=3, mistakes={"c'est la valeur par défaut d'Adam (0), pas d'AdamW": 0.0})

WB_ANSWER {"decimals": 3, "hash": "d8b77141cadd53333ed9b30932c2116ae1b9c30b70955d7d1ae9411298f3aac8", "hash_coarse": "38c95427d5ddc3a6e965494cbdfed2f3b3b631cf942a43838e7f547fef31333e", "id": "20.R1a", "kind": "float", "magnitude": 0, "mistakes": {"776dabf5f95d303fd759d03fe6fbc3073fd074748d6bdcec690ba05287d6c6d5": "le gradient est nul, mais le weight decay agit quand même", "a4e36621e526ca06cc1ef254b3b88276651c44ac5b7da04a2b2f7425b585ccc0": "le facteur est 1 − η λ, avec η = 0,01 : multiplie λ par η"}, "sign": 1}
📝 Ex 20.R1a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"hash": "f3dead4a97fd50c5fee73955d2d8ec62010cb848aa36492a235847441f4c5110", "id": "20.R1b", "kind": "bool", "mistakes": {"3031939fed7edf4acadcad5514d31faf549fdd5d289b0925d13adb9de5b8a1d9": "c'est la différence entre Adam(weight_decay=…) et AdamW : relis la fiche du ch. 19"}}
📝 Ex 20.R1b : réponse enregistrée (bool).
WB_ANSWER {"decimals": 3, "hash": "8701c65b07940e396fd01a0e28a28e6ad64ac0c22b34cdc745a8848892f2d

**20.R2 — Pourquoi calcule-t-on la loss à partir des logits ?**

In [16]:
wb.record("20.R2a", False, mistakes={"CrossEntropyLoss fait elle-même log_softmax : quelle entrée attend-elle ?": True})
wb.record("20.R2b", round(-1000 - math.log1p(math.exp(-1000))), mistakes={"log_softmax(z)_2 = z_2 − log(e^{z_1} + e^{z_2}) : le logarithme d'une probabilité est négatif": 1000,
          "log_softmax(z)_2 = z_2 − log(e^{z_1} + e^{z_2}) ≈ 0 − 1000": 0})
wb.record("20.R2c", 1)
wb.record("20.R2d", "B", mistakes={"le but n'est pas d'écrire moins de code": "A", "la softmax a une dérivée (ch. 17)": "C",
          "torch.softmax existe : pourquoi ne pas l'appliquer avant la loss ?": "D"})

WB_ANSWER {"hash": "2684913b17598ad6640b8035ae8cbbc0a22eed7042c46d6decb8317d7ce515e7", "id": "20.R2a", "kind": "bool", "mistakes": {"e37a7814901221188f40b90e48c7a80afd4626e23ab4df51994cd03874752970": "CrossEntropyLoss fait elle-même log_softmax : quelle entrée attend-elle ?"}}
📝 Ex 20.R2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "66b5a6e8c6588126be8210c12e7e677f07be24797519bfa4a65c65212596394b", "id": "20.R2b", "kind": "int", "magnitude": 3, "mistakes": {"b5a5653fc85c82da5586ece5913069d9770fdfddde410fa1254ba2a721a232f2": "log_softmax(z)_2 = z_2 − log(e^{z_1} + e^{z_2}) ≈ 0 − 1000", "db6256130b79d1f713298d2fc452c95503df1755426b358e9eb55a782b0dce2f": "log_softmax(z)_2 = z_2 − log(e^{z_1} + e^{z_2}) : le logarithme d'une probabilité est négatif"}, "sign": -1}
📝 Ex 20.R2b : réponse enregistrée (int).
WB_ANSWER {"hash": "45b8cb18157c5d5fd4352d442c1123b89a869e276c0c2600c55207da0d5e94d9", "id": "20.R2c", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 20.R2c : réponse enregistrée (int)

**20.R3 — Limiter l'overfitting d'un arbre, et l'équivalent pour un réseau**

In [17]:
wb.record("20.R3a", "A", mistakes={"la graine change l'arbre, pas sa complexité": "B", "n_jobs change la vitesse": "C",
          "le critère change le choix des seuils, pas la taille de l'arbre": "D"})
wb.record("20.R3b", "B", mistakes={"plus de neurones, c'est plus de capacité, donc plus d'overfitting": "A",
          "plus d'epochs laisse plus de temps pour apprendre par cœur": "C",
          "sans validation, on ne voit plus l'overfitting": "D"})
wb.record("20.R3c", True, mistakes={"l'early stopping arrête l'entraînement avant que l'erreur de validation ne remonte : que fait-il à l'overfitting ?": False})

WB_ANSWER {"hash": "3d77f9a76c186192a6e2d5e9d37cf7c33ee611b51b09f85031561704f27a6e8b", "id": "20.R3a", "kind": "str", "mistakes": {"c3394212c80baa12e4fef47aa500b5afe47d362d8a10e6a3171812ff9deabf86": "la graine change l'arbre, pas sa complexité", "cb83abb2faa57265702e53f1226f8fe1e4c09c76b3971f641a792eb388e33fe8": "n_jobs change la vitesse", "e257d07e7a185b0e20514c363dbdc9a8d823c957bdc44986648827359abec38c": "le critère change le choix des seuils, pas la taille de l'arbre"}}
📝 Ex 20.R3a : réponse enregistrée (str).
WB_ANSWER {"hash": "9a8aa07cc260d5e3cb8395db89d841683c7fc07b1f6efd88abcc1729f1af139c", "id": "20.R3b", "kind": "str", "mistakes": {"87485239f5eec89f5fa5aa2446e079818e289340b7486f2c843bfd1997209347": "plus de neurones, c'est plus de capacité, donc plus d'overfitting", "d002d9d020eebeed37fbb8fb5ff6e5d08afd437b8ae82bfdb237eb6d3251c85a": "plus d'epochs laisse plus de temps pour apprendre par cœur", "d48ef80dda7d6a0448a1c64e4c0f4e9d067dc5368e1285650e2396cd709df8b2": "sans validatio

**Ex 20.1 — Suivre la forme d'un tenseur couche après couche**

In [18]:
wb.record("20.1a", 10 * 3 * 5, mistakes={"le nombre d'éléments est le produit des tailles": 18})
wb.record("20.1b", "(10, 15)", mistakes={"on garde la première dimension, 10 : 150 éléments à répartir": "(150,)",
          "garde la première dimension et multiplie les deux autres": "(10, 8)",
          "le remodelage garde la première dimension, 10, en premier": "(15, 10)"})
wb.record("20.1c", "(32, 784)", mistakes={"nn.Flatten garde la dimension des exemples": "(25088,)",
          "nn.Flatten met à plat tout ce qui suit la dimension des exemples, canal compris": "(32, 1, 784)",
          "1 × 28 × 28 = 784": "(32, 28)"})
wb.record("20.1d", "(32, 128)", mistakes={"la couche dense remplace les 784 features par 128 sorties": "(32, 784)",
          "la dimension des exemples reste la première": "(128, 32)"})
wb.record("20.1e", 784 * 128 + 128, mistakes={"ajoute les 128 biais": 784 * 128, "un biais par sortie, pas par entrée": 784 * 128 + 784})
wb.record("20.1f", "(32, 3072)", mistakes={"3 × 32 × 32 = 3 072 : compte aussi les trois canaux": "(32, 1024)",
          "nn.Flatten garde la dimension des exemples": "(98304,)"})
wb.record("20.1g", "(25088,)", mistakes={"torch.flatten met TOUT à plat, la dimension des exemples comprise": "(1, 25088)",
          "une forme à une dimension s'écrit avec une virgule : (7,)": "(25088)",
          "512 × 7 × 7 = 25 088": "(512, 49)"})
wb.record("20.1h", "(1, 25088)", mistakes={"nn.Flatten garde la première dimension (start_dim=1)": "(25088,)",
          "512 × 7 × 7 = 25 088": "(1, 512, 49)"})

WB_ANSWER {"hash": "6885d8749b9d5df14e0c7fc9d646edca0fb325d639918e63c0a92fa5392f2ee3", "id": "20.1a", "kind": "int", "magnitude": 2, "mistakes": {"96b0eece7bf4b3b2d2d96c192a8be298a87c3f58f4244557e83b6cfd4b2bcf6a": "le nombre d'éléments est le produit des tailles"}, "sign": 1}
📝 Ex 20.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "bb12aef547bcdd76a532672a1618c73becaa82e56b2aea29ea767c2023a9a50b", "id": "20.1b", "kind": "str", "mistakes": {"182c27ec29ed392ef849e705102bce48c1126cba9eec6379de86f280249e59ed": "le remodelage garde la première dimension, 10, en premier", "66f5d87fa04537b764d334fdebe46071b385510eecf53890c0279cf5b659e02a": "garde la première dimension et multiplie les deux autres", "dfa2680e0dee2699f8325a852ecb4c13c0001b88d7c61d8c730c299e0a0806b6": "on garde la première dimension, 10 : 150 éléments à répartir"}}
📝 Ex 20.1b : réponse enregistrée (str).
WB_ANSWER {"hash": "9be7946ab902617b8e8a086f4b44a5c24862821500e280d84055f14fa8241ed8", "id": "20.1c", "kind": "str", "mista

**Ex 20.2 — Dimensionner la couche de sortie et choisir la loss**

In [19]:
wb.record("20.2a", 1, mistakes={"un prix : une seule valeur à prédire": 2})
wb.record("20.2b", "A", mistakes={"un prix n'est pas une probabilité : c'est une régression": "B",
          "un prix n'est pas une classe": "C"})
wb.record("20.2c", 26, mistakes={"un score par classe": 1})
wb.record("20.2d", "C", mistakes={"une régression ne convient pas pour 26 classes exclusives": "A",
          "les 26 classes sont exclusives : une seule lettre par image": "B"})
wb.record("20.2e", "B", mistakes={"une probabilité de spam, pas une valeur réelle quelconque": "A",
          "CrossEntropyLoss attend un score par classe, donc au moins deux sorties": "C"})
wb.record("20.2f", 5, mistakes={"une sortie par label, chacun présent ou non": 1,
          "les labels ne sont pas exclusifs : pas besoin d'une sortie par combinaison": 2 ** 5})
wb.record("20.2g", "B", mistakes={"chaque label est un oui ou non : une probabilité par label": "A",
          "CrossEntropyLoss suppose qu'une seule classe est vraie : ici, plusieurs labels peuvent l'être": "C"})
wb.record("20.2h", "C", mistakes={"deux scores, une classe vraie parmi deux : c'est une classification à K = 2 classes": "B",
          "c'est une classification": "A"})

WB_ANSWER {"hash": "96a1258122dd076ebb5e54c4f1cf2594ad7ffacb53143c6f04f34b92f735e494", "id": "20.2a", "kind": "int", "magnitude": 0, "mistakes": {"d705aa38fdcd2f493920aadbe64b7ef463c8f186ab146539febb86118624fedb": "un prix : une seule valeur à prédire"}, "sign": 1}
📝 Ex 20.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "8f5358de71762839afa4c042975ce959c3ee845b7d78a5ee30c15d373a3fb531", "id": "20.2b", "kind": "str", "mistakes": {"0df1154f510fc8db780f34f68508ab4ca918dc78fed154a030d0a723f02595ee": "un prix n'est pas une probabilité : c'est une régression", "dbb22e75a1b982ed18cd967649e00793ed0ccd68f00d410fc90ab14db4a2912f": "un prix n'est pas une classe"}}
📝 Ex 20.2b : réponse enregistrée (str).
WB_ANSWER {"hash": "ac283a1166a5e6ce85028c5152c22c73e228e705d98f18e95d8e7bcb6c3862f1", "id": "20.2c", "kind": "int", "magnitude": 1, "mistakes": {"2da6641adcd79eac1865093024082cf5dce65d9dc1d634a04b5d05de790502a2": "un score par classe"}, "sign": 1}
📝 Ex 20.2c : réponse enregistrée (int).
WB_ANS

**Ex 20.3 — Dropout inversé à la main**

In [20]:
wb.record("20.3a", round(0.2 * 100), mistakes={"20 % de 100 neurones sont coupés : combien en garde-t-on ?": 80})
wb.record("20.3b", int((U_203 >= P_203).sum()), mistakes={"on GARDE un neurone quand u ≥ p, on le coupe quand u < p": 2})
wb.record("20.3c", 1 / (1 - P_203), decimals=4, mistakes={"on divise par 1 − p, on ne multiplie pas": 0.6, "on divise par 1 − p, pas par p": 2.5})
wb.record("20.3d", OUT_203[1], decimals=4, mistakes={"multiplie par 1/(1 − p)": -1.0, "u₂ = 0,7 ≥ 0,4 : ce neurone est gardé": 0.0,
          "on divise par 1 − p = 0,6, pas par p": -2.5})
wb.record("20.3e", OUT_203.sum(), decimals=4, mistakes={"multiplie par 1/(1 − p) les neurones gardés": float(A_203[U_203 >= P_203].sum()),
          "on garde quand u ≥ p, pas quand u < p": float((A_203 * (U_203 < P_203)).sum() / (1 - P_203)),
          "seuls les neurones gardés comptent dans la somme": float(A_203.sum() / (1 - P_203))})
wb.record("20.3f", A_203.sum(), decimals=4, mistakes={"en moyenne sur les masques, le dropout inversé garde l'espérance de chaque activation": float(OUT_203.sum()),
          "chaque activation garde son espérance, sans facteur 1 − p": float((1 - P_203) * A_203.sum())})
wb.record("20.3g", MASK_203.sum(), decimals=4, mistakes={"le gradient passe par le même masque, avec le même facteur 1/(1 − p)": 3.0,
          "seuls les neurones gardés transmettent le gradient": 5 / (1 - P_203)})
wb.record("20.3h", True, mistakes={"en évaluation, le dropout inversé est l'identité": False})

WB_ANSWER {"hash": "2c4c9ee80ce167219f895796d8f476cc9ac559ca38ff73cb95319fd6b2a34448", "id": "20.3a", "kind": "int", "magnitude": 1, "mistakes": {"02179b177a1365326bee85a1867eb9122dd48c23b4e47dbc7ff7a6965fc75b35": "20 % de 100 neurones sont coupés : combien en garde-t-on ?"}, "sign": 1}
📝 Ex 20.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "d8ad278cbf0af638a75b9f83759efa45c57c57a5c54cc3f699e4ace6d8df0c09", "id": "20.3b", "kind": "int", "magnitude": 0, "mistakes": {"1e64ed00a98a9f02a49a695f5fb74497235e0d23eddd131f841ac882d74e820e": "on GARDE un neurone quand u ≥ p, on le coupe quand u < p"}, "sign": 1}
📝 Ex 20.3b : réponse enregistrée (int).
WB_ANSWER {"decimals": 4, "hash": "ce29f8a883249d97f5f576614ae6e6303ac44f06d75284286c80fa8410b6f101", "hash_coarse": "279a03e5ed3c85230d0b618860dd15f7a2df507814a381c1c9dbfe6a3ee402f5", "id": "20.3c", "kind": "float", "magnitude": 0, "mistakes": {"23bf2e9347c4a6c24178d398decbc7d1b2175efbed328b328ce1babd07fbf348": "on divise par 1 − p, pas par p"

**Ex 20.4 — Batchnorm à la main sur un mini-batch de quatre exemples**

In [21]:
wb.record("20.4a", MU_204[0], decimals=4, mistakes={"une moyenne par feature, donc par colonne": float(X_204[0].mean())})
wb.record("20.4b", MU_204[1], decimals=4, mistakes={"une moyenne par feature, donc par colonne": float(X_204[1].mean())})
wb.record("20.4c", VAR_204[0], decimals=4, mistakes={"pour normaliser, la variance biaisée : divise par 4, pas par 3": float(X_204[:, 0].var(ddof=1)),
          "la variance, pas l'écart-type": float(X_204[:, 0].std())})
wb.record("20.4d", VAR_204[1], decimals=4, mistakes={"pour normaliser, la variance biaisée : divise par 4, pas par 3": float(X_204[:, 1].var(ddof=1)),
          "la variance, pas l'écart-type": float(X_204[:, 1].std())})
wb.record("20.4e", XHAT_204[0, 0], decimals=4, mistakes={"divise par l'écart-type, la racine de la variance": -3 / 5,
          "soustrais d'abord la moyenne de la feature, 4": float((1 - 0) / np.sqrt(5)),
          "avec la variance non biaisée, ce n'est plus la normalisation de PyTorch": float(-3 / X_204[:, 0].std(ddof=1))})
wb.record("20.4f", Y_204[1, 1], decimals=4, mistakes={"y = γ x̂ + β : n'oublie pas β₂ = 1": float(Y_204[1, 1] - 1),
          "c'est x̂, avant l'échelle γ et le décalage β": float(XHAT_204[1, 1]),
          "γ₂ = 0,5 et β₂ = 1, pas l'inverse": float(XHAT_204[1, 1] + 0.5)})
wb.record("20.4g", True, mistakes={"la moyenne des x̂ d'une feature est nulle : que vaut celle de γ x̂ + β ?": False})
wb.record("20.4h", RUN_MEAN_204[0], decimals=4, mistakes={"PyTorch : (1 − ρ) × ancienne + ρ × nouvelle, avec ρ = 0,1": 3.6,
          "la statistique glissante ne prend que 10 % de la nouvelle moyenne": 4.0})
wb.record("20.4i", RUN_MEAN_204[1], decimals=4, mistakes={"PyTorch : (1 − ρ) × ancienne + ρ × nouvelle, avec ρ = 0,1": 9.0,
          "la statistique glissante ne prend que 10 % de la nouvelle moyenne": 10.0})
wb.record("20.4j", RUN_VAR_204[0], decimals=4, mistakes={"la variance glissante utilise la variance NON biaisée, 20/3": 0.9 + 0.1 * 5,
          "partir de σ²_run = 1, pas de 0": 0.1 * 20 / 3,
          "(1 − ρ) × ancienne + ρ × nouvelle, avec ρ = 0,1, pas l'inverse": 0.1 + 0.9 * 20 / 3})
wb.record("20.4k", EVAL_204[0], decimals=3, mistakes={"en évaluation, les statistiques GLISSANTES, pas celles du mini-batch": 0.0,
          "n'oublie pas γ₁ = 2": float(EVAL_204[0] / 2),
          "la racine de la variance glissante, pas la variance": float(2 * 3.6 / RUN_VAR_204[0])})

WB_ANSWER {"decimals": 4, "hash": "81d397c4db78e30f7cd0626166d773bdca5106b75c3074961292ce8350edb090", "hash_coarse": "59abab6648678a8435cf7973dc76a9b14b36ab005691df899ab8b9a709eaa839", "id": "20.4a", "kind": "float", "magnitude": 0, "mistakes": {"40c359035abfad9596060e970c979e139d5045ea23d9c2449cff347f622c81e7": "une moyenne par feature, donc par colonne"}, "sign": 1}
📝 Ex 20.4a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "a7d5b99d8aedd2fdae35d6eec9f6f5bbc967260cbeac2b44befaa5f2fdb83a6a", "hash_coarse": "3501fd095347f08831e080c1f48e6413e179d0482e3aae883495527cc813ef78", "id": "20.4b", "kind": "float", "magnitude": 1, "mistakes": {"c81b151e8d159a70bb1cdf4a325a09be9ce5a26871f607b5aa1c00d175c5a969": "une moyenne par feature, donc par colonne"}, "sign": 1}
📝 Ex 20.4b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "a26f91aec049d355f1e957d842c182440303be9b669be76b1df58e044e5ceb89", "hash_coarse": "db7ab11bf71c4ec5418c81bfce

**Ex 20.6 — Compter les 138 millions de paramètres de VGG16**

In [22]:
wb.record("20.6a", 224 // 2 ** 5, mistakes={"le zero-padding garde la taille ; seuls les 5 poolings la divisent par 2": 224,
          "5 poolings : 224 / 2⁵": 14})
wb.record("20.6b", VGG_CONV[0], mistakes={"(3² × 3 + 1) × 64 : n'oublie pas le biais de chaque filtre": 27 * 64,
          "chaque filtre a 3 × 3 poids PAR canal d'entrée": 10 * 64})
wb.record("20.6c", sum(VGG_CONV), mistakes={"ajoute les biais : un par filtre": sum(VGG_CONV) - sum(sum(b) for b in VGG_BLOCKS)})
wb.record("20.6d", VGG_FC[0], mistakes={"ajoute les 4 096 biais": 25088 * 4096, "l'entrée est l'aplatissement de 512 × 7 × 7 = 25 088 valeurs": 512 * 4096 + 4096})
wb.record("20.6e", VGG_TOTAL, mistakes={"ajoute les biais des couches denses": VGG_TOTAL - 4096 - 4096 - 1000})
wb.record("20.6f", 100 * sum(VGG_FC) / VGG_TOTAL, decimals=1, mistakes={"compte les trois couches denses, pas seulement la première": 100 * VGG_FC[0] / VGG_TOTAL,
          "la part des denses, pas celle des convolutions": 100 * sum(VGG_CONV) / VGG_TOTAL})
wb.record("20.6g", 4 * VGG_TOTAL / 1e6, decimals=1, mistakes={"4 octets par nombre float32": VGG_TOTAL / 1e6,
          "Mo de 10⁶ octets, pas Mio de 2²⁰": 4 * VGG_TOTAL / 2 ** 20})
wb.record("20.6h", "B", mistakes={"compare ta réponse de g) à 527,8 : quelle unité donne le plus petit nombre ?": "A",
          "en mégabits, le nombre serait 8 fois plus grand": "C",
          "les convolutions ne font qu'environ 59 Mo : le fichier contient tout": "D"})

WB_ANSWER {"hash": "66308b4699583d8c2dbb6acefd73cc88f563c99d7edee364f7f2c3c8ff1e3519", "id": "20.6a", "kind": "int", "magnitude": 0, "mistakes": {"d11a25b4f9514906e7ffb7fa102f007cd3a3a8f25510f44203a20b7984e63217": "5 poolings : 224 / 2⁵", "d8fac1f6f9f0fd7565dba95526b7aaa009b90f621850be7d261565f6cea771d1": "le zero-padding garde la taille ; seuls les 5 poolings la divisent par 2"}, "sign": 1}
📝 Ex 20.6a : réponse enregistrée (int).
WB_ANSWER {"hash": "631fac19deb82b90d1a2272b4b39e2c4b3552f1bf83027211b54ae219608063f", "id": "20.6b", "kind": "int", "magnitude": 3, "mistakes": {"2830e1d0776b2202ffec0dec31c9945b1e0bcc03e76252923959f408fd0e9490": "chaque filtre a 3 × 3 poids PAR canal d'entrée", "6afc65efa7bec0ed37bf933e880f803b1b3a1119932607faf2077a45aad35770": "(3² × 3 + 1) × 64 : n'oublie pas le biais de chaque filtre"}, "sign": 1}
📝 Ex 20.6b : réponse enregistrée (int).
WB_ANSWER {"hash": "784f5a995b9eee7146e61635d9aa0d5c0b5c7232a84c55e2468eb7c33a41fa3d", "id": "20.6c", "kind": "int", "m

**Ex 20.7 — Connexion résiduelle et LayerNorm à la main**

In [23]:
wb.record("20.7a", X_207.mean(), decimals=4)
wb.record("20.7b", X_207.var(), decimals=4, mistakes={"la layer norm utilise la variance biaisée : divise par 4": float(X_207.var(ddof=1)),
          "la variance, pas l'écart-type": float(X_207.std())})
wb.record("20.7c", LN_207[3], decimals=4, mistakes={"divise par l'écart-type, la racine de la variance": 3 / 3.5,
          "avec la variance non biaisée, ce n'est plus la layer norm de PyTorch": float(3 / X_207.std(ddof=1))})
wb.record("20.7d", float((LN_207 ** 2).sum()), decimals=4, mistakes={"la moyenne des carrés vaut 1 : et leur somme ?": 1.0, "c'est la somme des valeurs qui est nulle": 0.0})
wb.record("20.7e", True, mistakes={"la layer norm calcule ses statistiques sur les features de chaque exemple, seul": False})
wb.record("20.7f", 0.5 ** 30 / 1e-9, decimals=4, mistakes={"en multiple de 10⁻⁹ : écris le coefficient devant 10⁻⁹, pas la valeur": 0.5 ** 30,
          "c'est un produit de 30 facteurs 0,5": 0.5 * 30 / 1e-9})
wb.record("20.7g", math.ceil(math.log(0.01) / math.log(0.9)), mistakes={"0,9⁴³ vaut encore un peu plus de 0,01 : il en faut une de plus": 43,
          "cherche n tel que 0,9ⁿ < 0,01, avec des logarithmes": 10})
wb.record("20.7h", 1.01 ** 30, decimals=4, mistakes={"la dérivée d'un bloc résiduel est 1 + F′, pas F′": 0.01 ** 30,
          "c'est un produit de 30 facteurs 1,01, pas une somme": 1 + 30 * 0.01})

WB_ANSWER {"decimals": 4, "hash": "ac988bad4ff817bd079b988613c8a6a91cce12c5b7e75d8b2af05bc0314580e2", "hash_coarse": "60675690943ee14b4e09f020b06f72d927a4cce6a94182408094abba72581bf4", "id": "20.7a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 20.7a : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "012ec2e314554ef3b44567d1cee472e154420e40a9749aae9b5cf9b4bfb83445", "hash_coarse": "f76bcafa4f43f2ced9918f756ebc672bf58d48f196017a8932ddc5d1913f8fe5", "id": "20.7b", "kind": "float", "magnitude": 0, "mistakes": {"02a598b5795b606f9c7cbe689b8075a30a91551c017f946991eb62ce1a874eda": "la layer norm utilise la variance biaisée : divise par 4", "c0f120fbb00090995e7612183788debd2c195f13f593d752a8d714fc5a648080": "la variance, pas l'écart-type"}, "sign": 1}
📝 Ex 20.7b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 4, "hash": "c16aa299d673860606550296b8b5b6a294a0a9b34d42c1a77411116009f7d62d", "hash_coarse": "056530a5b828ef6a55eed946256465154412f

**Ex 20.9 — Fermi : entraîner VGG16 sur ImageNet chez soi ?**

In [24]:
wb.record("20.9a", TRAIN_OPS_209 / 1e9, decimals=1, mistakes={"passe avant ET passe arrière : 3 fois le coût de la passe avant": 15.5,
          "3 fois plus en tout, pas 3 fois plus en plus": 4 * 15.5})
wb.record("20.9b", TOTAL_OPS_209 / 1e18, decimals=2, mistakes={"le coût d'entraînement est 3 fois celui de la passe avant": 1.28e6 * 74 * 15.5e9 / 1e18,
          "74 epochs sur 1,28 million d'images : multiplie par les deux": 1.28e6 * TRAIN_OPS_209 / 1e18})
wb.record("20.9c", TOTAL_OPS_209 / 1e13 / 86400, decimals=1, mistakes={"une journée a 86 400 secondes": TOTAL_OPS_209 / 1e13 / 3600,
          "le résultat en jours, pas en secondes": TOTAL_OPS_209 / 1e13})
wb.record("20.9d", round(TOTAL_OPS_209 / 1e11 / 86400), fractional="un nombre de jours, arrondi à l'entier",
          mistakes={"une journée a 86 400 secondes": round(TOTAL_OPS_209 / 1e11 / 3600)})
wb.record("20.9e", 4 * 4 * 138_357_544 / 1e9, decimals=2, mistakes={"paramètres, gradients ET les deux états d'Adam : 4 nombres par paramètre": 4 * 138_357_544 / 1e9,
          "4 nombres par paramètre, de 4 octets chacun": 3 * 4 * 138_357_544 / 1e9,
          "Go de 10⁹ octets, pas Gio": 4 * 4 * 138_357_544 / 2 ** 30})

WB_ANSWER {"alt_coarse": ["a55bd46007dd89c9e5da02707f259e855b1ab9501dc55178d400901f75e62a34"], "decimals": 1, "hash": "b90c67dd41df4d9ac09c846ec9bf656e883de292db3d640d5e932831de3cae65", "hash_coarse": "808966cc2fbb512f9a835d9643368f99250fb12cc603363ee80a531b70798c7f", "id": "20.9a", "kind": "float", "magnitude": 1, "mistakes": {"5c877c89734ba3aadef5503cde348e457f8aa0bb5a8258827eb7350932d39cef": "passe avant ET passe arrière : 3 fois le coût de la passe avant", "d4953508a279e253cc2645227ebf491b6d99b751917178fd178e317e4bc26605": "3 fois plus en tout, pas 3 fois plus en plus"}, "sign": 1}
📝 Ex 20.9a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "b369e142fe847a2a3b1b4ac8208dc61daf8817e2ecde5c23204276fa6af16167", "hash_coarse": "a20a47d00bc9c469083625a4692cd08baab7465afc43cf544b330cd98550501c", "id": "20.9b", "kind": "float", "magnitude": 0, "mistakes": {"0bde3141bb0882b3214d0cec9569cf4a5943a1bc4ba6d0b1e449ff3e5028321e": "le coût d'entraînement est 3 fois c

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu prévoir la forme d'un tenseur après une couche dense, un aplatissement ou un pooling, et choisir la sortie et la loss d'un réseau ?
2. Sais-tu calculer à la main un dropout inversé et une batchnorm, et dire ce que changent `model.train()` et `model.eval()` ?
3. Sais-tu écrire une boucle d'entraînement PyTorch complète et expliquer chacune de ses lignes ?

**Pour aller plus loin** : le tutoriel « Learn the Basics » de PyTorch et *Dive into Deep Learning*, cités dans la fiche. La suite : le ch. 21, les réseaux de convolution, en PyTorch.

*Les exercices de code 20.13 à 20.31 (parties A à D, ta librairie `regularization.py` et PyTorch) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 20` pour obtenir le notebook complet.*